# Script to parse jedi files

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import datetime as dt

import matplotlib.pyplot as plt 
import matplotlib.colors as colors
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection

from JunoPy.jedi import JediData, merge_data_species
from JunoPy.index import IndexFile
from JunoPy.library import find_ticks_between

In [ ]:
startevent = dt.datetime.strptime('2017-236T20:21:15', '%Y-%jT%H:%M:%S')
endevent   = dt.datetime.strptime('2017-236T20:21:38', '%Y-%jT%H:%M:%S')
start = startevent - dt.timedelta(minutes=20)
end   = endevent + dt.timedelta(minutes=20)

# jedifiles = glob.glob('data/jedi/*SISP_CDR_{:s}_*.TAB'.format(start.strftime('%Y%j')))
# if not jedifiles:
jediindex = IndexFile(dataset='jedi')
jediindex.get_files_between_dates(
    start, end, DoDownload=True, sid='ERSISP')
jediindex.get_files_between_dates(
    start, end, DoDownload=True, sid='ERSESP')
    
jedifiles = glob.glob('data/jedi/*_CDR_{:s}_*.TAB'.format(start.strftime('%Y%j')))
jedidatas = []
for jedifile in jedifiles:
    jedidatas.append(JediData(filename=jedifile))
    print(jedifile)

In [ ]:
# Separate ion files and electron files.
ion_data = []
electron_data = []
for jdata in jedidatas:
    
    jdata.df = jdata.df.loc[start:end]
    jdata.calc_broadband_flux()

    if jdata.metadata['particletype'].str.contains('ELECTRON').any():
        electron_data.append(jdata)
    else:
        ion_data.append(jdata)

ion_data[0].metadata.head()

In [ ]:
# Merge all data of same species.
data_merge = merge_data_species(ion_data, species='ion')
data_merge.df.head()

In [ ]:
fig = plt.figure(dpi=150)
ax = plt.subplot()
data_merge.plot_flux_spectra(ax)
ax.set_xlim(start, end)
ax.label_outer()
plt.show()

In [ ]:
# Plot pitch angle spectra.
def plot_pitchangle_spectra(data, n=2):
    """
    Type Plot -
            P_PitchAngle, P_EvsTime, Heavy_PitchAngle, Heavy_EvsTime
    """
    chars = ['x', 'y', 'z']
    timearray, tickstring = find_ticks_between(data.df.index[0], data.df.index[-1])
    # timearray, tickstring = find_ticks_between(start, end)
    
    fig = plt.figure(dpi=150)
    ax = plt.subplot()
    
    data.plot_pitchangle_scatter(ax=ax)
    ax.set_xlim(timearray[0], timearray[-1])
    ax.set_xticks(timearray)
    ax.set_xticklabels(tickstring)
    ax.label_outer()
    # ax.xaxis.set_minor_locator(AutoMinorLocator())

    plt.show()

print([col for col in data_merge.df if col.startswith('BroadbandFlux')])
    
plot_pitchangle_spectra(data_merge, n=2)

In [ ]:
# df.plot(y=['Pitch_Angle_T{:d}'.format(iax) for iax in range(6)], figsize=(11,8.5), legend=True, markersize=11., style='.')

def plot_pitchangle_spectra(data, start=None, end=None, n=2):
    """
    Type Plot -
            P_PitchAngle, P_EvsTime, Heavy_PitchAngle, Heavy_EvsTime
    """
    chars = ['x', 'y', 'z']
    
    if start and end:
        data.df = data.df.loc[start:end]
        
    timearray, tickstring = find_ticks_between(data.df.index[0], data.df.index[-1])
    
    fig = plt.figure(dpi=150)
    ax = plt.subplot(111) 
    
    xticks = data.plot_pitchangle_spectra(ax=ax)
    
    ax.set_xlim(xticks[0], xticks[-1])
    ax.set_xticks(xticks)
    ax.set_xticklabels(tickstring)
    ax.label_outer()

    plt.show()

plot_pitchangle_spectra(data_merge)

